# DUBAI TOWER — Floor 03: Semantic Room-Circulation Graph

Semantic room-graph for the Dubai Tower **Level 03** apartments, using the Modified Swiss
Dwellings (MSD) graph-ML node schema. This notebook builds and **visualises** the graph only
(no GNN prediction):

1. Load each **room type** OBJ as closed **cells** (rooms = graph nodes, typed).
2. Connect rooms through the **doors** (`aperture.obj`) — a door in the wall between two
   rooms creates a **circulation edge** (matched by proximity, since real walls have
   thickness so room volumes don't share faces).
3. Build the **semantic room-circulation graph** and visualise it (plan view).

> Prediction is intentionally out of scope here — see `DubaiTower_F04_Semantic_Prediction.ipynb`
> for the GNN room-type classifier on Floor 04.

## 1. Imports

In [1]:
import os, random
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from collections import Counter

random.seed(42); np.random.seed(42)
renderer = "vscode"

## 2. Configuration — paths, room types, MSD feature mappings

In [2]:
from pathlib import Path

# OBJ folder (rooms, doors, windows for Floor 03)
FLOOR_TAG = "F03"
def _find_obj_dir(tag):
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / "assets" / "obj" / tag
        if (p / "bedroom.obj").exists():
            return p
    raise FileNotFoundError(f"Could not find assets/obj/{tag} with bedroom.obj")
OBJ_DIR = _find_obj_dir(FLOOR_TAG)
print("OBJ_DIR:", OBJ_DIR)

# Room-type OBJ files present for Floor 03 (filename = semantic label)
ROOM_FILES = {
    "bedroom":    "bedroom.obj",
    "livingroom": "livingroom.obj",
    "kitchen":    "kitchen.obj",
    "corridor":   "corridor.obj",
    "stairs":     "stairs.obj",
    "bathroom":   "bathroom.obj",
    "storeroom":  "storeroom.obj",
    "balcony":    "balcony.obj",
}
DOOR_FILE   = "aperture.obj"   # interior doors -> circulation edges
WINDOW_FILE = "window.obj"     # exterior openings -> per-room feature

# --- MSD label mapping (reference, 9 classes) -------------------------------
ROOM_LABEL = {"bedroom":0,"livingroom":1,"kitchen":2,"dining":3,"corridor":4,
              "stairs":5,"storeroom":6,"bathroom":7,"balcony":8}
# Display colours per room type
ROOM_COLOR = {"bedroom":"#FFBFBF","bathroom":"#4444FF","corridor":"#7FFFBF","kitchen":"#BF3F3F",
              "livingroom":"#FFBF00","stairs":"#BF3FFF","storeroom":"#FF7FFF","dining":"#A0522D",
              "balcony":"#007F00","unknown":"#AAAAAA"}

PROXIMITY_TOL = 0.6   # max distance (m) from a door/window centroid to a room cell to count as touching

OBJ_DIR: c:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\F03


## 3. Load room OBJs as closed cells (the graph nodes)

Each room type OBJ is imported with `transposeAxes=True` (so plan is in X-Y, height in Z) and
each room object is rebuilt into a watertight **Cell** via `Cell.ByFaces` at progressive
tolerances. The room type (from the filename) is stored on every cell.

In [3]:
def build_cell(faces):
    for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
        c = Cell.ByFaces(faces, tolerance=tol)
        if c is not None:
            return c
    return None

cells, rtypes = [], []
for rtype, fn in ROOM_FILES.items():
    path = str(OBJ_DIR / fn)
    if not os.path.exists(path):
        print(f"  [SKIP] missing {fn}"); continue
    objs = Topology.ByOBJPath(path, transposeAxes=True)
    if not isinstance(objs, list): objs = [objs]
    n = 0
    for obj in objs:
        if obj is None: continue
        faces = Topology.Faces(obj) or []
        if len(faces) < 4: continue
        c = build_cell(faces)
        if c is None: continue
        d = Dictionary.ByKeysValues(["room_type", "color"], [rtype, ROOM_COLOR.get(rtype, "#AAAAAA")])
        c = Topology.SetDictionary(c, d)
        cells.append(c); rtypes.append(rtype); n += 1
    print(f"  {rtype:11s}: {n} cells")

N = len(cells)
centroids = [Topology.Centroid(c) for c in cells]
cen_xyz = np.array([[Vertex.X(v), Vertex.Y(v), Vertex.Z(v)] for v in centroids])
print(f"\nTotal rooms (nodes): {N}")
print("Distribution:", dict(Counter(rtypes)))

present = sorted(set(rtypes), key=lambda t: ROOM_LABEL[t])
print("Room types present:", present)

  bedroom    : 20 cells
  livingroom : 9 cells
  kitchen    : 9 cells
  corridor   : 15 cells
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
  stairs     : 3 cells
  bathroom   : 26 cells
  storeroom  : 10 cells
  balcony 

## 4. Visualise the room cells coloured by type

In [4]:
fig = Topology.Show(cells, faceColorKey="color", faceOpacity=0.85,
                    showEdges=True, edgeColor="black", edgeWidth=1,
                    showVertices=False, backgroundColor="white",
                    width=1100, height=750, showFigure=False, renderer=renderer)
fig.update_layout(
    title=dict(text="Floor 03 — rooms coloured by type", font=dict(color="black")),
    paper_bgcolor="white",
    scene=dict(aspectmode="data",
               xaxis=dict(color="black"), yaxis=dict(color="black"), zaxis=dict(color="black")),
    scene_camera=dict(projection=dict(type="orthographic"), eye=dict(x=0, y=0, z=2.2)))
fig.show(renderer=renderer)

# colour legend
print("Room-type colours:")
for t in present:
    print(f"  {t:11s} {ROOM_COLOR[t]}")

Room-type colours:
  bedroom     #FFBFBF
  livingroom  #FFBF00
  kitchen     #BF3F3F
  corridor    #7FFFBF
  stairs      #BF3FFF
  storeroom   #FF7FFF
  bathroom    #4444FF
  balcony     #007F00


## 5. Load doors + windows, and build the circulation edges

Doors (`aperture.obj`) and windows are flat faces. Real walls have thickness, so room volumes
don't share faces — instead each **door** is matched by **proximity** to the two nearest room
cells; those two rooms get a circulation edge. Each **window** is matched to its single nearest
room (a per-room feature).

In [5]:
def load_faces(fn):
    path = str(OBJ_DIR / fn)
    if not os.path.exists(path):
        print(f"  [SKIP] missing {fn}"); return []
    objs = Topology.ByOBJPath(path, selfMerge=False)
    if not isinstance(objs, list): objs = [objs]
    faces = []
    for obj in objs:
        if obj is None: continue
        fs = Topology.Faces(obj) or []
        if fs:
            faces.extend(fs)
        else:
            for w in (Topology.Wires(obj) or []):
                f = Face.ByWire(w)
                if f is None:
                    w2 = Topology.RemoveCollinearEdges(w); f = Face.ByWire(w2) if w2 else None
                if f is not None: faces.append(f)
    return faces

doors   = load_faces(DOOR_FILE)
windows = load_faces(WINDOW_FILE)
print(f"doors: {len(doors)}   windows: {len(windows)}")

def nearest_rooms(face, k, maxd):
    fc = Topology.Centroid(face)
    ds = sorted((Vertex.Distance(fc, cells[i]), i) for i in range(N))
    return [i for d, i in ds if d <= maxd][:k]

# Door -> edge between its 2 nearest rooms
edge_set = set()
unmatched_doors = 0
for d in doors:
    near = nearest_rooms(d, 2, PROXIMITY_TOL)
    if len(near) >= 2:
        a, b = sorted(near[:2])
        if a != b: edge_set.add((a, b))
    else:
        unmatched_doors += 1
edges = sorted(edge_set)

# Window -> nearest room (feature)
win_count = [0] * N
for w in windows:
    near = nearest_rooms(w, 1, PROXIMITY_TOL)
    if near: win_count[near[0]] += 1

deg = Counter()
for a, b in edges: deg[a] += 1; deg[b] += 1
isolated = [i for i in range(N) if deg[i] == 0]

# Adjacency fallback: a balcony is accessed from the room it adjoins, but if its
# access door isn't in aperture.obj it would be left isolated. Connect every still-
# isolated room to its nearest room (by centroid distance) so the graph stays connected.
ADJ_FALLBACK_TOL = 8.0
_c2 = cen_xyz[:, :2]
_extra = []
for i in isolated:
    d = ((_c2 - _c2[i])**2).sum(1); d[i] = 1e18
    j = int(d.argmin())
    if d[j]**0.5 <= ADJ_FALLBACK_TOL:
        _extra.append(tuple(sorted((i, j))))
if _extra:
    edge_set |= set(_extra); edges = sorted(edge_set)
    deg = Counter()
    for a, b in edges: deg[a] += 1; deg[b] += 1
    isolated = [i for i in range(N) if deg[i] == 0]
    print(f"adjacency fallback: +{len(set(_extra))} edge(s); isolated now {len(isolated)}")
print(f"circulation edges: {len(edges)}   doors not bridging 2 rooms: {unmatched_doors}")
print(f"connected rooms: {N - len(isolated)}/{N}   isolated: {len(isolated)}")
et = Counter(tuple(sorted((rtypes[a], rtypes[b]))) for a, b in edges)
print("\nTop adjacency types:")
for pair, c in et.most_common(10):
    print(f"  {pair[0]:11s} - {pair[1]:11s}: {c}")

doors: 106   windows: 22
adjacency fallback: +6 edge(s); isolated now 0
circulation edges: 102   doors not bridging 2 rooms: 6
connected rooms: 101/101   isolated: 0

Top adjacency types:
  bedroom     - corridor   : 19
  corridor    - livingroom : 18
  bathroom    - corridor   : 17
  corridor    - storeroom  : 10
  balcony     - livingroom : 9
  corridor    - corridor   : 6
  bathroom    - bedroom    : 5
  kitchen     - livingroom : 4
  bathroom    - livingroom : 3
  corridor    - kitchen    : 3


## 6. Visualise the semantic room-circulation graph (plan view)

In [6]:
fig = go.Figure()
# edges
ex, ey = [], []
for a, b in edges:
    ex += [cen_xyz[a,0], cen_xyz[b,0], None]
    ey += [cen_xyz[a,1], cen_xyz[b,1], None]
fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines",
                         line=dict(color="rgba(80,80,80,0.5)", width=1.2),
                         hoverinfo="skip", showlegend=False))
# nodes by room type
for t in present:
    idx = [i for i in range(N) if rtypes[i] == t]
    fig.add_trace(go.Scatter(
        x=cen_xyz[idx,0], y=cen_xyz[idx,1], mode="markers",
        marker=dict(size=12, color=ROOM_COLOR[t], line=dict(color="black", width=1)),
        name=f"{t} ({len(idx)})",
        text=[f"{t} (deg {deg[i]}, win {win_count[i]})" for i in idx], hoverinfo="text"))
fig.update_layout(
    title=dict(text="Floor 03 — semantic room-circulation graph", font=dict(color="black")),
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(color="black", showgrid=True, gridcolor="rgba(0,0,0,0.08)", scaleanchor="y", scaleratio=1),
    yaxis=dict(color="black", showgrid=True, gridcolor="rgba(0,0,0,0.08)"),
    legend=dict(font=dict(color="black")), width=1100, height=750)
fig.show(renderer=renderer)